# DL · Encuentro 4 · Detección, evaluación y revisión

Detección y decisiones con evidencia

**Tu copia personal:** en Colab elige **Archivo → Guardar una copia en Drive**. Conserva esa copia para los tres temas del encuentro.

Trabaja en CPU. Estas prácticas no requieren contratar GPU, claves de API ni servicios de pago. La primera preparación puede descargar bibliotecas ausentes; las prácticas de detección descargan los pesos del modelo si no están disponibles.

## Tu recorrido

1. [De una foto a un registro](#scrollTo=tema-dl-yolo)
2. [Una imagen marcada no prueba calidad](#scrollTo=tema-dl-evaluacion)
3. [Del modelo a una decisión responsable](#scrollTo=tema-dl-integracion)

Cada tema reúne **idea → caso explicado → preparación → ejecución → lectura y comprobación**. Puedes ejecutar todo el cuaderno o empezar por un tema: cada uno vuelve a preparar sus datos y herramientas. No mezcles las cifras de conjuntos diferentes.

Los datos inventados se identifican como simulaciones. Las fotos de detección son demostraciones y no incluyen una evaluación anotada de sus objetos. Las comprobaciones son voluntarias y sin nota.


## Tema 1 · De una foto a propuestas y registros

**Lo que vas a poder hacer:** ¿Cómo seguimos un objeto propuesto desde la foto hasta un registro que podamos revisar?

Usaremos bus.jpg, una foto incluida en la biblioteca, y un detector preentrenado con categorías de COCO. Bus y persona pertenecen a su catálogo; defectos del café no se aprenden al cambiar un nombre.

### Predice antes de ejecutar

Antes de pedir predicciones, mira la foto y nombra dos objetos que buscarías. Después predice qué cambiará al elevar el umbral: ¿los píxeles originales o las propuestas conservadas?

### Ejecuta y compara

1. Ejecuta las celdas de este tema en orden: preparación, catálogo, foto, detección, tabla, conteo, umbrales y guardado. Lee la foto original antes de ejecutar la detección.
2. Elige una caja y encuentra una fila con la misma categoría y coordenadas aproximadas. Compara después las tres fotos de umbrales 0,15, 0,50 y 0,85: identifica una propuesta que permanezca o desaparezca, además del total.

### Explica con tus palabras

«La fila propone… en la zona… con puntuación…; el corte… la conserva/descarta. Todavía falta comprobar…».

**Cómo conecta con el aula:** La web usa puntuaciones dadas para aprender el filtro. Aquí el modelo calcula propuestas sobre una foto real. El registro guarda propuestas conservadas, todavía no un inventario confirmado.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere los temas anteriores. Si repites el tema desde cero, comienza por su preparación.

**Qué cambiar y qué conservar:** Conserva foto y modelo; compara únicamente los umbrales del bloque. Tanto esta biblioteca como el nuevo ejercicio de la web conservan puntuaciones estrictamente mayores que el corte. Igualar el corte no basta para conservar una propuesta.


### Herramientas que vas a usar

**YOLO** (You Only Look Once, solo se mira una vez) es una familia de detectores. **COCO** (Common Objects in Context, objetos comunes en contexto) es el conjunto de categorías del modelo usado. **CPU** es el procesador; esta práctica no necesita GPU. **RGB** significa rojo, verde y azul: son los tres canales de la foto.

Ejecuta la preparación y mira la foto original antes de pedir propuestas.


In [ ]:
import sys, subprocess
from importlib import metadata

VERSION_ULTRALYTICS = '8.4.175'
try:
    version_instalada = metadata.version('ultralytics')
except metadata.PackageNotFoundError:
    version_instalada = None
if version_instalada != VERSION_ULTRALYTICS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           f'ultralytics=={VERSION_ULTRALYTICS}'])
import ultralytics
if ultralytics.__version__ != VERSION_ULTRALYTICS:
    raise RuntimeError('Ultralytics sigue cargado con otra versión. Reinicia el entorno y ejecuta de nuevo desde la preparación.')
print('Ultralytics:', ultralytics.__version__)


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, FileLink
import torch
from ultralytics import YOLO
from ultralytics.utils import ASSETS

SALIDA = Path('resultados_yolo')
SALIDA.mkdir(exist_ok=True)
torch.set_num_threads(2)
TRADUCCION = {'person':'persona','bus':'bus','car':'automóvil','truck':'camión',
              'bicycle':'bicicleta','motorcycle':'motocicleta','chair':'silla',
              'bottle':'botella','cup':'taza','laptop':'portátil','cell phone':'teléfono',
              'book':'libro','backpack':'mochila','tie':'corbata'}
def nombre_es(nombre):
    return TRADUCCION.get(nombre, nombre)
print('PyTorch:',torch.__version__)
print('Los nombres no traducidos conservan la categoría original en inglés.')


In [ ]:
modelo = YOLO('yolo26n.pt')
categorias = pd.DataFrame([{'id':i,'clase_original':nombre,'nombre':nombre_es(nombre)}
                           for i,nombre in modelo.names.items()])
print('Categorías conocidas:',len(categorias))
print('Ejemplos del catálogo:',', '.join(categorias.nombre.head(15)))
print('Son solo ejemplos; una categoría no mostrada aún puede existir. La búsqueda consulta el catálogo completo.')
BUSCAR_CATEGORIA='café' # después prueba 'persona' sin cambiar los pesos
consulta=BUSCAR_CATEGORIA.strip().casefold()
coincidencias=categorias[(categorias.nombre.str.casefold().str.contains(consulta,regex=False))|
                         (categorias.clase_original.str.casefold().str.contains(consulta,regex=False))] if consulta else categorias.iloc[:0]
if coincidencias.empty:
    print('No hay coincidencias para',repr(BUSCAR_CATEGORIA),'en este catálogo. No renombres otra clase para fingir que la reconoce.')
else:
    print('Coincidencias en todo el catálogo:',', '.join(coincidencias.nombre))


In [ ]:
foto = ASSETS / 'bus.jpg'
original = Image.open(foto).convert('RGB')
print('Ancho y alto:',original.size,'· canales: RGB')
fig,ax=plt.subplots(figsize=(5,7))
ax.imshow(original);ax.axis('off');ax.set_title('Entrada: la foto sin predicciones')
plt.tight_layout();plt.savefig(SALIDA/'01_original.png',dpi=130);plt.show()


In [ ]:
UMBRAL = 0.25
resultado = modelo.predict(source=str(foto), conf=UMBRAL, imgsz=640,
                           device='cpu', verbose=False)[0]
print('Detecciones conservadas:',len(resultado.boxes))
resultado.save(filename=str(SALIDA/'02_foto_detectada.jpg'))
fig,axs=plt.subplots(1,2,figsize=(10,7))
axs[0].imshow(original);axs[0].set_title('Antes: píxeles')
axs[1].imshow(resultado.plot()[:,:,::-1]);axs[1].set_title('Después: categorías, cajas y puntuaciones')
for ax in axs:ax.axis('off')
plt.tight_layout();plt.savefig(SALIDA/'03_antes_despues.png',dpi=130);plt.show()


In [ ]:
def tabla_detecciones(r):
    columnas=['deteccion','clase_id','clase_original','categoria','puntuacion',
              'x1','y1','x2','y2','ancho_px','alto_px']
    filas=[]
    for i,caja in enumerate(r.boxes, start=1):
        clase=int(caja.cls.item());nombre=r.names[clase]
        x1,y1,x2,y2=caja.xyxy[0].cpu().numpy().tolist()
        filas.append([i,clase,nombre,nombre_es(nombre),float(caja.conf.item()),
                      x1,y1,x2,y2,x2-x1,y2-y1])
    return pd.DataFrame(filas,columns=columnas)

tabla = tabla_detecciones(resultado)
display(tabla.round(3))
assert len(tabla)==len(resultado.boxes)
if not tabla.empty:
    primera=tabla.iloc[0]
    print('Primera detección:',primera.categoria,
          '· puntuación:',round(primera.puntuacion,3),
          '· tamaño:',round(primera.ancho_px),'×',round(primera.alto_px),'px')


In [ ]:
def resumir(t):
    return (t.groupby('categoria').size().rename('detecciones').reset_index()
             .sort_values('detecciones',ascending=False))
resumen=resumir(tabla)
display(resumen)
fig,ax=plt.subplots(figsize=(7,3))
if not resumen.empty:
    ax.bar(resumen.categoria,resumen.detecciones,color='#0075ad')
    ax.set_yticks(range(0,int(resumen.detecciones.max())+1))
else:ax.text(.5,.5,'Sin detecciones conservadas',ha='center',transform=ax.transAxes)
ax.set_ylabel('Detecciones en esta foto');ax.set_title('Un registro derivado de las cajas')
plt.tight_layout();plt.savefig(SALIDA/'04_conteo.png',dpi=130);plt.show()


In [ ]:
umbrales=[0.15,0.50,0.85]
comparacion=[]
fig,axs=plt.subplots(1,3,figsize=(12,6))
for ax,u in zip(axs,umbrales):
    r=modelo.predict(str(foto),conf=u,imgsz=640,device='cpu',verbose=False)[0]
    ax.imshow(r.plot()[:,:,::-1]);ax.axis('off')
    ax.set_title(f'Umbral {u:.2f}: {len(r.boxes)} detecciones')
    comparacion.append({'umbral':u,'detecciones':len(r.boxes)})
plt.tight_layout();plt.savefig(SALIDA/'05_umbrales.png',dpi=130);plt.show()
display(pd.DataFrame(comparacion))


### Guarda qué se propuso y con qué configuración

**CSV** (valores separados por comas) guarda las filas de detecciones y conteos como tablas. **JSON** (notación de objetos de JavaScript) guarda nombres y valores de la configuración, como modelo y umbral. Guardar archivos permite revisar la ejecución; no demuestra que las detecciones sean correctas.


In [ ]:
tabla.to_csv(SALIDA/'detecciones.csv',index=False,encoding='utf-8-sig')
resumen.to_csv(SALIDA/'conteo.csv',index=False,encoding='utf-8-sig')
config={'modelo':'yolo26n.pt','ultralytics':ultralytics.__version__,
        'torch':torch.__version__,'foto':foto.name,'umbral':UMBRAL,'imgsz':640,
        'dispositivo':'cpu','entrenamiento_en_este_cuaderno':False}
(SALIDA/'configuracion.json').write_text(json.dumps(config,indent=2,ensure_ascii=False))
for p in [SALIDA/'02_foto_detectada.jpg',SALIDA/'detecciones.csv',SALIDA/'conteo.csv',SALIDA/'configuracion.json']:
    print('Archivo creado:',p)
    display(FileLink(str(p)))


### Lee, comprueba y explica

**Qué debes poder comprobar:** Elige una caja y encuentra una fila con la misma categoría y coordenadas aproximadas. Compara después las tres fotos de umbrales 0,15, 0,50 y 0,85: identifica una propuesta que permanezca o desaparezca, además del total.

**Prueba pequeña:** Conserva foto y modelo; compara únicamente los umbrales del bloque. Tanto esta biblioteca como el nuevo ejercicio de la web conservan puntuaciones estrictamente mayores que el corte. Igualar el corte no basta para conservar una propuesta.

**Pregunta:** ¿Cómo seguimos un objeto propuesto desde la foto hasta un registro que podamos revisar?

**Respuesta razonada:** El corte filtra propuestas, conserva los píxeles de entrada y no entrena de nuevo. Cada fila corresponde a una propuesta conservada. Para comprobar calidad hacen falta referencias anotadas; la foto marcada y su conteo no bastan.

**Registra en tu copia:** «La fila propone… en la zona… con puntuación…; el corte… la conserva/descarta. Todavía falta comprobar…». No es una entrega ni una calificación.

[Volver al tema del aula](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-yolo)

[Siguiente tema: Una imagen marcada no prueba calidad](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-evaluacion)


## Tema 2 · Comprobar cajas con una referencia conocida

**Lo que vas a poder hacer:** ¿Una propuesta conservada está bien localizada o deja un objeto sin detectar?

Cambiamos de la foto real a dos cajas inventadas con referencia conocida. Ambas miden 40×40 píxeles y tienen la misma categoría. La propuesta vale 0,80. Esta práctica evalúa esas cajas; no calcula la calidad del detector sobre bus.jpg.

### Predice antes de ejecutar

Con el corte de puntuación en 0,50 y la IoU mínima en 0,50, ¿desplazar la propuesta 30 píxeles puede producir una falsa propuesta y una omisión a la vez? Dibuja qué quedaría sin emparejar.

### Ejecuta y compara

1. Ejecuta la preparación de este tema y la evaluación de cajas. Observa los tres escenarios de la misma figura.
2. Primero compara desplazamientos de 10 y 30 píxeles: área compartida e IoU. Después compara 10 píxeles con corte 0,50 y con corte 0,90: la ubicación es la misma, pero la propuesta se filtra. Justifica cada TP, FP y FN con una caja concreta.

### Explica con tus palabras

«La propuesta se conserva/descarta porque…; coincide/no coincide porque…; la referencia queda…; por eso los conteos son…».

**Cómo conecta con el aula:** El tema anterior obtenía propuestas sin anotaciones. Ahora añadimos una referencia inventada para aprender a comprobar ubicación. IoU mide superposición; la puntuación del modelo sirve para filtrar. Son números diferentes.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere los temas anteriores. Si repites el tema desde cero, comienza por su preparación.

**Qué cambiar y qué conservar:** Conserva inicialmente los tres escenarios del código. Compara una sola diferencia cada vez. Una referencia se empareja como máximo una vez.


### Prepara este tema de forma independiente

Ejecuta las dos celdas de preparación aunque no hayas abierto el tema anterior. La evaluación siguiente utiliza cajas dadas: no necesita entrenar un modelo ni evaluar objetos de una foto.


In [ ]:
import sys, subprocess
from importlib import metadata

VERSION_ULTRALYTICS = '8.4.175'
try:
    version_instalada = metadata.version('ultralytics')
except metadata.PackageNotFoundError:
    version_instalada = None
if version_instalada != VERSION_ULTRALYTICS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           f'ultralytics=={VERSION_ULTRALYTICS}'])
import ultralytics
if ultralytics.__version__ != VERSION_ULTRALYTICS:
    raise RuntimeError('Ultralytics sigue cargado con otra versión. Reinicia el entorno y ejecuta de nuevo desde la preparación.')
print('Ultralytics:', ultralytics.__version__)


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, FileLink
import torch
from ultralytics import YOLO
from ultralytics.utils import ASSETS

SALIDA = Path('resultados_yolo')
SALIDA.mkdir(exist_ok=True)
torch.set_num_threads(2)
TRADUCCION = {'person':'persona','bus':'bus','car':'automóvil','truck':'camión',
              'bicycle':'bicicleta','motorcycle':'motocicleta','chair':'silla',
              'bottle':'botella','cup':'taza','laptop':'portátil','cell phone':'teléfono',
              'book':'libro','backpack':'mochila','tie':'corbata'}
def nombre_es(nombre):
    return TRADUCCION.get(nombre, nombre)
print('PyTorch:',torch.__version__)
print('Los nombres no traducidos conservan la categoría original en inglés.')


### Construye la superposición antes de contar errores

La referencia es [10,20,50,60]: esquina izquierda-superior (10,20) y derecha-inferior (50,60). Ancho=50−10=40 píxeles; alto=60−20=40; área=40×40=1600 píxeles cuadrados. La propuesta mide lo mismo.

Para desplazar **10 píxeles** a la derecha:

1. La propuesta queda en [20,20,60,60].
2. El ancho común es 50−20=30; alto común=40; área común=30×40=1200.
3. Sumamos áreas y restamos lo común para no contarlo dos veces: 1600+1600−1200=2000.
4. **IoU** (intersección sobre unión)=área común÷área de la unión=1200÷2000=0,60. No tiene unidades.

Para desplazar **30 píxeles**: ancho común=10; área común=10×40=400; unión=1600+1600−400=2800; IoU=400÷2800≈0,143.

La puntuación 0,80 se compara con el corte para conservar la propuesta. Después, la categoría correcta y una IoU de al menos 0,50 permiten emparejarla con la referencia.

- **TP, verdadero positivo:** propuesta y referencia emparejadas correctamente.
- **FP, falso positivo:** propuesta conservada sin coincidencia válida.
- **FN, falso negativo:** objeto de referencia que quedó sin una propuesta válida.

Una propuesta mal ubicada puede generar un FP y dejar un FN al mismo tiempo. Esta cuenta pequeña no calcula AP ni mAP, métricas agregadas que requieren otro protocolo y más anotaciones.


In [ ]:
# Datos inventados para aprender evaluación, independientes de las fotos y de resultado.
from matplotlib.patches import Rectangle
def iou_cajas(a,b):
    izquierda=max(a[0],b[0]);arriba=max(a[1],b[1])
    derecha=min(a[2],b[2]);abajo=min(a[3],b[3])
    comun=max(0,derecha-izquierda)*max(0,abajo-arriba)
    area_a=max(0,a[2]-a[0])*max(0,a[3]-a[1])
    area_b=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    union=area_a+area_b-comun
    return comun/union if union else 0.

referencia=np.array([10.,20.,50.,60.]) # ancho 40 y alto 40 píxeles
PUNTUACION_EJEMPLO=.80;MINIMO_IOU=.50
escenarios=[('Desplazar 10 px',10,.50),('Desplazar 30 px',30,.50),('Filtrar con 0,90',10,.90)]
fig,axs=plt.subplots(1,3,figsize=(12,4));filas=[]
for ax,(nombre,desplazamiento,limite_puntuacion) in zip(axs,escenarios):
    propuesta=referencia+np.array([desplazamiento,0,desplazamiento,0])
    iou=iou_cajas(referencia,propuesta)
    conservada=PUNTUACION_EJEMPLO>limite_puntuacion
    tp=int(conservada and iou>=MINIMO_IOU);fp=int(conservada and not tp);fn=1-tp
    ax.add_patch(Rectangle(referencia[:2],40,40,color='#0075ad',alpha=.3,label='Referencia'))
    ax.add_patch(Rectangle(propuesta[:2],40,40,fill=False,edgecolor='#b85d00',lw=2,
                           linestyle='-' if conservada else '--',label='Propuesta' if conservada else 'Descartada'))
    comun_x=max(referencia[0],propuesta[0]);comun_ancho=max(0,min(referencia[2],propuesta[2])-comun_x)
    if comun_ancho:ax.add_patch(Rectangle((comun_x,20),comun_ancho,40,color='#198a87',alpha=.4,label='Área común'))
    ax.set(xlim=(0,95),ylim=(80,0),xlabel='x: píxeles desde la izquierda',ylabel='y: píxeles desde arriba')
    ax.set_aspect('equal');ax.set_title(f'{nombre}\nIoU={iou:.3f} · TP/FP/FN={tp}/{fp}/{fn}',fontsize=10);ax.legend(fontsize=7)
    filas.append([iou,tp,fp,fn])
    print(nombre,'· puntuación',PUNTUACION_EJEMPLO,'· límite',limite_puntuacion,
          '· conservada',conservada,'· aciertos/falsas propuestas/omisiones:',tp,fp,fn)
plt.tight_layout();plt.savefig(SALIDA/'06_evaluacion_cajas.png',dpi=130);plt.show()
assert np.isclose(filas[0][0],.6) and np.isclose(filas[1][0],1/7)
if MINIMO_IOU==.50 and PUNTUACION_EJEMPLO==.80:
    assert filas[0][1:]==[1,0,0] and filas[1][1:]==[0,1,1] and filas[2][1:]==[0,0,1]
else:
    print('Configuración cambiada: interpreta los nuevos conteos con el criterio que elegiste.')
print('IoU desplazamiento 10: área común 30×40=1200; unión 1600+1600−1200=2000; 1200/2000=0,60.')
print('IoU desplazamiento 30: área común 10×40=400; unión 1600+1600−400=2800; 400/2800=1/7.')


### Lee, comprueba y explica

**Qué debes poder comprobar:** Primero compara desplazamientos de 10 y 30 píxeles: área compartida e IoU. Después compara 10 píxeles con corte 0,50 y con corte 0,90: la ubicación es la misma, pero la propuesta se filtra. Justifica cada TP, FP y FN con una caja concreta.

**Prueba pequeña:** Conserva inicialmente los tres escenarios del código. Compara una sola diferencia cada vez. Una referencia se empareja como máximo una vez.

**Pregunta:** ¿Una propuesta conservada está bien localizada o deja un objeto sin detectar?

**Respuesta razonada:** Con la configuración inicial, los tres escenarios producen TP/FP/FN de 1/0/0, 0/1/1 y 0/0/1. La propuesta mal ubicada es falsa y deja sin detectar la referencia. Filtrarla elimina esa falsa propuesta, pero la referencia sigue omitida.

**Registra en tu copia:** «La propuesta se conserva/descarta porque…; coincide/no coincide porque…; la referencia queda…; por eso los conteos son…». No es una entrega ni una calificación.

[Volver al tema del aula](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-evaluacion)

[Siguiente tema: Del modelo a una decisión responsable](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-integracion)


## Tema 3 · Comparar costes de revisión

**Lo que vas a poder hacer:** ¿Qué regla de revisión conviene si revisar y dejar pasar defectos tienen consecuencias diferentes?

Tenemos diez granos ficticios con puntuaciones dadas de defecto. Aquí 1 significa defectuoso y 0 significa sin defecto. Revisar cuesta 1 punto; aceptar un defecto cuesta inicialmente 8. Suponemos revisión perfecta; los puntos no son dinero real.

### Predice antes de ejecutar

Con corte 0,50 se revisan cuatro granos y se acepta un defecto. Separa la cuenta en coste de revisar + coste de omitir. ¿Podría salir más caro que revisar los diez?

### Ejecuta y compara

1. Ejecuta la preparación de este tema y la tabla de costes. Guarda la comparación inicial. Cambia solo COSTO_DEFECTO_ACEPTADO de 8 a 1 y vuelve a ejecutar la celda completa.
2. Para los cortes 0,30 y 0,50, compara revisiones, defectos omitidos y coste. Después compara las dos ejecuciones: las dos primeras cantidades no cambian; cambia su valoración. Si hay empate, compara también cuántas revisiones exige cada regla.

### Explica con tus palabras

«Con estos costes prefiero… porque supone… revisiones y… defectos aceptados. Si cambia…, la comparación…».

**Cómo conecta con el aula:** Esta es otra tarea: decidir revisión de unidades conocidas mediante puntuaciones de defecto dadas. Estas puntuaciones no son la confianza de presencia de YOLO ni salen del detector de autobuses.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere los temas anteriores. Si repites el tema desde cero, comienza por su preparación.

**Qué cambiar y qué conservar:** Mantén los diez granos, sus etiquetas, sus puntuaciones y el coste de revisión. Cambia solo el coste de omisión. Las etiquetas conocidas sirven para evaluar; una regla real se elige con validación y se comprueba después con prueba reservada.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### Separa decisión y consecuencia

En esta celda **real_defecto=1 significa defectuoso** y 0 significa sin defecto. Las puntuaciones p_defecto están dadas; no provienen de YOLO ni de una red entrenada aquí. La etiqueta real sirve para comprobar después qué defecto se omitió.

Para el corte 0,50:

1. Se revisa si p_defecto≥0,50. Eso selecciona los granos 2, 5, 7 y 8: **4 revisiones**.
2. Entre los aceptados queda el grano 4, que era defectuoso: **1 defecto omitido**.
3. Coste de revisar=4 revisiones×1 punto por revisión=**4 puntos**.
4. Coste de omitir=1 defecto×8 puntos por defecto=**8 puntos**.
5. Total=coste de revisar+coste de omitir=4+8=**12 puntos**.

No se cuenta como omitido un defecto revisado: suponemos que la revisión lo identifica correctamente. Esta revisión perfecta y estos costes son supuestos del ejercicio, no resultados medidos de una operación.


### Compara dos valoraciones de los mismos casos

Ejecuta con COSTO_DEFECTO_ACEPTADO=8 y conserva la tabla. Después cambia solo ese coste a 1 y vuelve a ejecutar la celda. Aquí positivo significa defectuoso, aunque otros ejemplos del curso usaran 1 para sano.


In [ ]:
# Positivo = defectuoso. Una puntuación alta dispara revisión manual.
real_defecto=np.array([0,1,0,1,1,0,0,1,0,0])
p_defecto=np.array([.1,.8,.2,.4,.9,.1,.6,.7,.2,.3])
COSTO_REVISION=1.;COSTO_DEFECTO_ACEPTADO=8.
filas=[]
for umbral in [.3,.5,.7,.9]:
    revisar=p_defecto>=umbral
    costo=revisar.sum()*COSTO_REVISION+np.sum((~revisar)&(real_defecto==1))*COSTO_DEFECTO_ACEPTADO
    filas.append([umbral,revisar.sum(),np.sum((~revisar)&(real_defecto==1)),costo])
resultado=pd.DataFrame(filas,columns=['Umbral','Granos revisados','Defectos omitidos','Costo ilustrativo']);display(resultado)
plt.plot(resultado.Umbral,resultado['Costo ilustrativo'],'o-',color=AZUL);plt.axhline(len(real_defecto)*COSTO_REVISION,color=VERDE,ls='--',label='Revisar todos');plt.xlabel('Umbral para revisar');plt.ylabel('Puntos de coste supuestos');plt.title('La decisión de negocio también tiene un parámetro');plt.legend();mostrar('14-impacto')
print('Supuesto del ejercicio: la revisión manual detecta todos los defectos. En un proyecto real, esa revisión también debe evaluarse.')


### Después de ejecutar: explica el coste y la carga de trabajo

Con coste de omisión 8, el corte 0,30 produce 6 revisiones×1+0 omisiones×8=**6 puntos**. El corte 0,50 produce 4×1+1×8=**12 puntos**. Revisar todos cuesta 10×1=**10 puntos**.

Con coste de omisión 1, el corte 0,70 revisa 3 y omite 1: 3×1+1×1=**4**. El corte 0,90 revisa 1 y omite 3: 1×1+3×1=**4**. Empatan en puntos, pero requieren distintas revisiones y aceptan distintos defectos.

Las puntuaciones y etiquetas se conservaron; cambió una consecuencia. En una aplicación real habría que medir capacidad, costes y errores de la revisión. La validación ayuda a elegir la regla y una prueba reservada permite comprobarla después; estas diez filas didácticas no hacen ambas funciones de manera independiente.


### Lee, comprueba y explica

**Qué debes poder comprobar:** Para los cortes 0,30 y 0,50, compara revisiones, defectos omitidos y coste. Después compara las dos ejecuciones: las dos primeras cantidades no cambian; cambia su valoración. Si hay empate, compara también cuántas revisiones exige cada regla.

**Prueba pequeña:** Mantén los diez granos, sus etiquetas, sus puntuaciones y el coste de revisión. Cambia solo el coste de omisión. Las etiquetas conocidas sirven para evaluar; una regla real se elige con validación y se comprueba después con prueba reservada.

**Pregunta:** ¿Qué regla de revisión conviene si revisar y dejar pasar defectos tienen consecuencias diferentes?

**Respuesta razonada:** Con coste de omisión 8, el corte 0,30 cuesta 6 puntos y el 0,50 cuesta 12; revisar todos cuesta 10. Al cambiar a 1 se conserva cuántos granos revisa y omite cada regla, pero cambia el total. Compara también carga de trabajo y supuestos, especialmente ante un empate.

**Registra en tu copia:** «Con estos costes prefiero… porque supone… revisiones y… defectos aceptados. Si cambia…, la comparación…». No es una entrega ni una calificación.

[Volver al tema del aula](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-integracion)


## Cierra el encuentro con una explicación

1. **De una foto a un registro:** El filtro cambia qué se conserva, no lo que aprendió el modelo ni los objetos presentes.
2. **Una imagen marcada no prueba calidad:** El umbral de puntuación decide qué propuestas revisar; el de IoU comprueba ubicación. Son cortes distintos.
3. **Del modelo a una decisión responsable:** La alternativa preferida depende de las consecuencias declaradas, además del modelo. La revisión perfecta es un supuesto que habría que comprobar.

En cada resultado identifica entrada, salida, unidad o significado, forma de comprobarlo y límite. Guarda tu copia personal. Colab no registra automáticamente tu avance en el aula.
